# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

In [1]:
!pip install razdel pymorphy3 spacy nltk wordcloud pandas matplotlib regex
!python -m spacy download ru_core_news_sm

     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     - -------------------------------------- 0.4/15.3 MB 9.2 MB/s eta 0:00:02
     -- ------------------------------------- 1.0/15.3 MB 10.5 MB/s eta 0:00:02
     ----- ---------------------------------- 2.1/15.3 MB 14.5 MB/s eta 0:00:01
     ------- -------------------------------- 3.1/15.3 MB 16.2 MB/s eta 0:00:01
     ---------- ----------------------------- 4.0/15.3 MB 17.0 MB/s eta 0:00:01
     ------------ --------------------------- 5.0/15.3 MB 17.5 MB/s eta 0:00:01
     --------------- ------------------------ 5.9/15.3 MB 18.9 MB/s eta 0:00:01
     ------------------ --------------------- 7.0/15.3 MB 18.6 MB/s eta 0:00:01
     -------------------- ------------------- 7.8/15.3 MB 20.0 MB/s eta 0:00:01
     -------------------- ------------------- 7.8/15.3 MB 20.0 MB/s eta 0:00:01
     --------------------- ------------------ 8.1/15.3 MB 17.2 MB/s eta 0:00:01
     --------------------- ------------------ 8.1

In [1]:
import os

import regex as re
from collections import Counter

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

from razdel import tokenize
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words("russian"))

import spacy
nlp = spacy.load("ru_core_news_sm")

import pandas as pd

In [2]:
print("Текущая папка:", os.getcwd())

Текущая папка: C:\Users\User\Downloads


In [36]:
file_path = "erdman.txt"

with open(file_path, "r", encoding="utf-8") as f:
    raw = f.read()

In [48]:
text = raw.replace("\r\n", "\n").replace("\r", "\n")
text = re.sub(r"\[\d+\]", "", text)

In [49]:
months = {
    "января": 1, "январь": 1, "янв": 1,
    "февраля": 2, "февраль": 2, "фев": 2,
    "марта": 3, "март": 3, "мар": 3,
    "апреля": 4, "апрель": 4, "апр": 4,
    "мая": 5, "май": 5,
    "июня": 6, "июнь": 6, "июн": 6,
    "июля": 7, "июль": 7, "июл": 7,
    "августа": 8, "август": 8, "авг": 8,
    "сентября": 9, "сентябрь": 9, "сен": 9,
    "октября": 10, "октябрь": 10, "окт": 10,
    "ноября": 11, "ноябрь": 11, "ноя": 11,
    "декабря": 12, "декабрь": 12, "дек": 12
}

roman_months = {
    "i": 1, "ii": 2, "iii": 3, "iv": 4,
    "v": 5, "vi": 6, "vii": 7, "viii": 8,
    "ix": 9, "x": 10, "xi": 11, "xii": 12
}

In [50]:
def parse_date(s, last_year=None):
    s = s.lower()

    m = re.search(r"(\d{1,2})[./](\d{1,2})[./-](\d{2,4})", s)
    if m:
        d = int(m.group(1))
        mo = int(m.group(2))
        y = int(m.group(3))
        if y < 100:
            y = 1900 + y
        return f"{y:04d}-{mo:02d}-{d:02d}", y

    m = re.search(r"(\d{1,2})[./]([ivx]{1,4})[./-](\d{2,4})", s)
    if m:
        d = int(m.group(1))
        mo = roman_months.get(m.group(2))
        y = int(m.group(3))
        if y < 100:
            y = 1900 + y
        if mo:
            return f"{y:04d}-{mo:02d}-{d:02d}", y

    m = re.search(r"(\d{1,2})[./](\d{1,2})-(\d{2,4})", s)
    if m:
        d = int(m.group(1))
        mo = int(m.group(2))
        y = int(m.group(3))
        if y < 100:
            y = 1900 + y
        return f"{y:04d}-{mo:02d}-{d:02d}", y

    m = re.search(r"(\d{1,2})\s+([а-яё]+)\.?\s+(\d{2,4})", s)
    if m:
        d = int(m.group(1))
        mo = months.get(m.group(2))
        y = int(m.group(3))
        if y < 100:
            y = 1900 + y
        if mo:
            return f"{y:04d}-{mo:02d}-{d:02d}", y

    m = re.search(r"(\d{1,2})\s+([а-яё]+)", s)
    if m and last_year is not None:
        d = int(m.group(1))
        mo = months.get(m.group(2))
        if mo:
            return f"{last_year:04d}-{mo:02d}-{d:02d}", None

    m = re.search(r"([а-яё]+)\.?\s+(\d{2,4})", s)
    if m:
        mo = months.get(m.group(1))
        y = int(m.group(2))
        if y < 100:
            y = 1900 + y
        if mo:
            return f"{y:04d}-{mo:02d}-01", y

    m = re.search(r"\b([а-яё]+)\b", s)
    if m and last_year is not None:
        mo = months.get(m.group(1))
        if mo:
            return f"{last_year:04d}-{mo:02d}-01", None

    m = re.search(r"\b(19\d{2}|20\d{2})\b", s)
    if m:
        y = int(m.group(1))
        return f"{y:04d}-01-01", y

    return None, None

In [51]:
parts = re.split(r"(?:\*\s*){3,}", text)

print("Всего блоков между ***:", len(parts))

Всего блоков между ***: 75


In [52]:
def clean_block(block, last_year):
    
    block = re.sub(r"\[Воспоминания.*?Предчувствие", "Предчувствие", block, flags=re.S)

    lines = block.split("\n")
    body = []
    started = False

    header_words = (
        r"^(Москва|Баку|Батум|Томск|Заказное|Енисейск|"
        r"проезд|Художественный|Ангелине|Ольге|для А|А\. О\.|"
        r"ул\.|Эрдман|До востребования|Оперный|Гастроли|"
        r"Чаква|Цихис|Деревня|Большая Мурта|Телеграмма|Начальнику)"
    )

    for line in lines:
        l = line.strip()

        if not started:
            if l == "":
                continue

            if re.search(header_words, l):
                continue

            date, y = parse_date(l, last_year)
            if date is not None:
                continue

            started = True

        body.append(line)

    result = "\n".join(body)
    result = re.sub(r"\[\d+\]", "", result)
    result = re.sub(r"\s+", " ", result)
    return result.strip()

In [53]:
def get_author(block):
    tail = block[-500:].lower()

    if re.search(r"\bниколай\b|\bколя\b|\bн\.", tail):
        return "Эрдман"

    if re.search(r"\bлина\b|\bангелина\b", tail):
        return "Степанова"

    return None

In [54]:
def lemmatize_text(txt):
    tokens = [t.text.lower() for t in tokenize(txt)]

    words = []
    for t in tokens:
        if re.fullmatch(r"[а-яёa-z0-9]+", t):
            words.append(t)

    lemmas = []
    for w in words:
        lemma = morph.parse(w)[0].normal_form
        lemmas.append(lemma)

    return " ".join(lemmas), len(words)

In [55]:
rows = []
last_year = None

for block in parts:
    block = block.strip()

    if block == "":
        continue

    if block.startswith("[Воспоминания"):
        continue

    author = get_author(block)
    if author is None:
        continue

    header = "\n".join(block.split("\n")[:15])
    date, found_year = parse_date(header, last_year)

    if found_year is not None:
        last_year = found_year

    body = clean_block(block, last_year)

    if len(body) == 0:
        continue

    lemma_text, word_count = lemmatize_text(body)

    rows.append({
        "date": date,
        "author": author,
        "text": body,
        "lemmatized_text": lemma_text,
        "word_count": word_count
    })

df = pd.DataFrame(rows)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date", na_position="last").reset_index(drop=True)

print("Извлечено писем:", len(df))
df.head(10)

Извлечено писем: 68


,date,author,text,lemmatized_text,word_count
0,1933-11-03,Эрдман,Только что задавал корм своему Гнедко (Петушок...,только что задавать корм свой гнедко петушок с...,114
1,1933-11-04,Степанова,"Дорогой, прекрасный, любимый, мне хочется, что...",дорога прекрасный любимый я хотеться чтобы это...,122
2,1933-11-05,Степанова,"Дорогой мой, посылка халтурно составлена, наск...",дорога мой посылка халтурно составить наскоро ...,28
3,1933-11-06,Эрдман,Застрял в деревне. На почтовых уехать не удало...,застрять в деревня на почтовый уехать не удать...,183
4,1933-11-06,Степанова,"Николашенька, вчера пришла телеграмма, что теб...",николашенек вчера прийти телеграмма что ты ост...,131
5,1933-11-20,Эрдман,Сюрпризы кончаются рано. К четырем годам их у ...,сюрприз кончаться рано к четыре год они у я уж...,485
6,1933-11-22,Эрдман,Вот это жизнь — сюрприз за сюрпризом. Сегодня ...,вот это жизнь сюрприз за сюрприз сегодня решит...,354
7,1933-11-22,Степанова,"Вчера получила твое письмо из Мурты, невыразим...",вчера получить твой письмо из мурот невыразимы...,181
8,1933-11-23,Степанова,Сегодня обитатели дома Лужских были удивлены. ...,сегодня обитатель дом лужский быть удивить пос...,124
9,1933-11-24,Степанова,А я все-таки получила от тебя подарок в день р...,а я получить от ты подарок в день рождение вче...,197


In [57]:
df = pd.DataFrame(rows)

df = df.sort_values("date").reset_index(drop=True)

df

,date,author,text,lemmatized_text,word_count
0,1933-11-03,Эрдман,Только что задавал корм своему Гнедко (Петушок...,только что задавать корм свой гнедко петушок с...,114
1,1933-11-04,Степанова,"Дорогой, прекрасный, любимый, мне хочется, что...",дорога прекрасный любимый я хотеться чтобы это...,122
2,1933-11-05,Степанова,"Дорогой мой, посылка халтурно составлена, наск...",дорога мой посылка халтурно составить наскоро ...,28
3,1933-11-06,Эрдман,Застрял в деревне. На почтовых уехать не удало...,застрять в деревня на почтовый уехать не удать...,183
4,1933-11-06,Степанова,"Николашенька, вчера пришла телеграмма, что теб...",николашенек вчера прийти телеграмма что ты ост...,131
...,...,...,...,...,...
63,None,Эрдман,Сегодня получил Твою телеграмму из Москвы и тр...,сегодня получить твой телеграмма из москва и т...,130
64,None,Эрдман,6 нояб. 34 г. Бездомная моя барышня. Твои грус...,6 ноябнуть 34 г бездомный мой барышня твой гру...,114
65,None,Эрдман,"Кажется, моя первая ночь в Томске продолжалась...",казаться мой первый ночь в томск продолжаться ...,191
66,None,Эрдман,Пришлось и мне перейти на открытки — в городе ...,прийтись и я перейти на открытка в город нет к...,182


In [58]:
print(df.info())
print()
print(df["author"].value_counts())
print()
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   date             54 non-null     object
 1   author           68 non-null     object
 2   text             68 non-null     object
 3   lemmatized_text  68 non-null     object
 4   word_count       68 non-null     int64 
dtypes: int64(1), object(4)
memory usage: 2.8+ KB
None

Эрдман       43
Степанова    25
Name: author, dtype: int64

         date     author                                               text  \
0  1933-11-03     Эрдман  Только что задавал корм своему Гнедко (Петушок...   
1  1933-11-04  Степанова  Дорогой, прекрасный, любимый, мне хочется, что...   
2  1933-11-05  Степанова  Дорогой мой, посылка халтурно составлена, наск...   
3  1933-11-06     Эрдман  Застрял в деревне. На почтовых уехать не удало...   
4  1933-11-06  Степанова  Николашенька, вчера пришла телеграмма, что теб

In [59]:
df.to_csv("erdman_stepanova_letters.csv", index=False, encoding="utf-8-sig")

print("Файл сохранён: erdman_stepanova_letters.csv")

Файл сохранён: erdman_stepanova_letters.csv
